# Stationarity & ARIMA

*Based on: Hyndman & Athanasopoulos, Forecasting: Principles and Practice (3rd ed.), Chapter 9*

---


## Learning Objectives

At the end of this notebook, you should be able to:

- Define **stationarity** in plain words and explain why most classical
  time-series models require it.
- Run the **ADF** and **KPSS** tests and read their p-values correctly
  (they have *opposite* null hypotheses!).
- Apply first and **seasonal differencing** to remove trend and seasonality.
- Recognise **AR(p)** and **MA(q)** patterns in ACF/PACF plots.
- Fit an **ARIMA(p, d, q)** model with statsmodels and inspect its residuals.
- Use **`auto_arima`** to search for an order automatically (and recognise when not to rely on it).


## Setup

`pmdarima.auto_arima` does a stepwise search over $(p, d, q)$, fast on
small data, but it still adds a few seconds of work. Under `NB_FAST=1`
(the CI default) we skip the search and instead fit a known-good order
directly.


In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from statsmodels.datasets import get_rdataset
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss

import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

NB_FAST = os.environ.get("NB_FAST", "").lower() in {"1", "true", "yes"}
warnings.filterwarnings("ignore")  # KPSS prints small-sample warnings


Load AirPassengers and a log-transformed copy. The log stabilises the
growing seasonal swing and is the version we will actually model.


In [ ]:
# AirPassengers, monthly, 1949-01 .. 1960-12.
air_raw = get_rdataset("AirPassengers").data
air = air_raw["value"].copy()
air.index = pd.date_range("1949-01-01", periods=len(air), freq="MS")
air.name = "passengers"

# A log-transformed copy stabilises the growing seasonal swing.
log_air = np.log(air)
log_air.name = "log_passengers"

print(f"air     : mean={air.mean():.1f}, std={air.std():.1f}")
print(f"log_air : mean={log_air.mean():.3f}, std={log_air.std():.3f}")


## 4.1 What does "stationary" mean?

A series is **stationary** if its statistical properties, mean, variance,
and autocorrelation structure, **do not depend on the time at which the
series is observed**. In pictures: the series should look roughly the same
whether you zoom in on its first half or its second half.

Most classical models (AR, MA, ARMA, ARIMA) **assume stationarity**.
If your series is not stationary, you transform it until it is, typically
by **differencing** and sometimes by taking logs.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
air.plot(ax=axes[0], color="seagreen")
axes[0].set_title("Air passengers: NOT stationary (trend + growing seasonal)")

# White noise is the canonical stationary series.
rng = np.random.default_rng(seed=0)
white_noise = pd.Series(rng.normal(size=200), index=pd.date_range("2000-01-01", periods=200, freq="D"))
white_noise.plot(ax=axes[1], color="slategrey")
axes[1].set_title("White noise: stationary by construction")

for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()


## 4.2 Testing for stationarity: ADF and KPSS

Two tests are widely used. **Read their null hypotheses carefully, they
are opposite!**

| Test | Null hypothesis ($H_0$) | Small p-value → |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | series **has a unit root** (non-stationary) | reject → series IS stationary |
| **KPSS** | series is **stationary** | reject → series is NOT stationary |

A good practice is to **run both** and check they agree.


In [ ]:
def stationarity_report(series, name):
    """Run ADF + KPSS and pretty-print the verdict."""
    adf_p = adfuller(series.dropna(), autolag="AIC")[1]
    kpss_p = kpss(series.dropna(), regression="c", nlags="auto")[1]
    print(f"{name}")
    print(f"  ADF  p={adf_p:.3f}  -> {'stationary' if adf_p < 0.05 else 'NOT stationary'}")
    print(f"  KPSS p={kpss_p:.3f}  -> {'NOT stationary' if kpss_p < 0.05 else 'stationary'}")

stationarity_report(air, "air (raw)")
stationarity_report(white_noise, "white noise")


The raw air-passengers series is flagged as non-stationary by both tests
(ADF fails to reject, KPSS rejects). White noise passes both. Now we need
to *make* `air` stationary so we can model it.


## 4.3 Differencing: the standard fix

The **first difference** is $\Delta y_t = y_t - y_{t-1}$ and removes a
linear trend. The **seasonal difference** is
$\Delta_m y_t = y_t - y_{t-m}$ and removes seasonality with period $m$.

For air passengers we use both:

1. log-transform to stabilise the variance, then
2. apply one seasonal difference at lag 12, then
3. apply one first difference.


In [ ]:
# log -> seasonal diff (lag 12) -> first diff.
log_air_d12 = log_air.diff(12)
log_air_d12_d1 = log_air_d12.diff(1)

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
log_air.plot(ax=axes[0], color="seagreen")
axes[0].set_title("log(air)")

log_air_d12.plot(ax=axes[1], color="steelblue")
axes[1].set_title("log(air) after seasonal diff at lag 12")

log_air_d12_d1.plot(ax=axes[2], color="crimson")
axes[2].set_title("log(air) after seasonal diff + first diff  (this is what we model)")

plt.tight_layout()
plt.show()


Re-run the stationarity report on the original and the doubly-differenced
series to confirm the difference operations did the job.


In [ ]:
stationarity_report(log_air, "log(air)")
stationarity_report(log_air_d12_d1, "log(air) differenced (d=1, D=1, m=12)")


After the two differences both tests agree the series is stationary,
this is what we will feed to ARIMA. The number of times we differenced
becomes the **$d$** (and seasonal **$D$**) parameter of the model.


## 4.4 AR(p): autoregression

An **AR(p)** model predicts the next value as a linear combination of the
last $p$ values:

$$y_t = \phi_1 y_{t-1} + \phi_2 y_{t-2} + \ldots + \phi_p y_{t-p} + \epsilon_t$$

**Diagnostic rule of thumb:**
A pure AR(p) shows an **ACF that decays slowly** and a **PACF that cuts off sharply after lag p**.


In [ ]:
# Simulate an AR(2) and look at its ACF/PACF.
rng = np.random.default_rng(seed=1)
n = 400
phi1, phi2 = 0.6, -0.3
ar = np.zeros(n)
for t in range(2, n):
    ar[t] = phi1 * ar[t - 1] + phi2 * ar[t - 2] + rng.normal()
ar = pd.Series(ar, index=pd.date_range("2000-01-01", periods=n, freq="D"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
plot_acf(ar, lags=20, ax=axes[0])
plot_pacf(ar, lags=20, ax=axes[1], method="ywm")
axes[0].set_title("ACF of an AR(2): decays slowly")
axes[1].set_title("PACF of an AR(2): cuts off at lag 2")
plt.tight_layout()
plt.show()


## 4.5 MA(q): moving-average errors

This is distinct from the moving-average smoother of notebook 3. An
**MA(q)** model writes today's value as a sum of the last $q$ *forecast
errors*:

$$y_t = \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + \ldots + \theta_q \epsilon_{t-q} + \epsilon_t$$

**Diagnostic rule of thumb:**
The shapes flip, **ACF cuts off at lag q**, **PACF decays slowly**.


In [ ]:
# Simulate an MA(1) and look at its ACF/PACF.
n = 400
theta = 0.7
eps = rng.normal(size=n)
ma = eps.copy()
for t in range(1, n):
    ma[t] = eps[t] + theta * eps[t - 1]
ma = pd.Series(ma, index=pd.date_range("2000-01-01", periods=n, freq="D"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
plot_acf(ma, lags=20, ax=axes[0])
plot_pacf(ma, lags=20, ax=axes[1], method="ywm")
axes[0].set_title("ACF of an MA(1): cuts off at lag 1")
axes[1].set_title("PACF of an MA(1): decays slowly")
plt.tight_layout()
plt.show()


## 4.6 ARIMA(p, d, q): putting it together

**ARIMA** stitches the three letters into one model:

- **AR(p)**: the autoregressive piece.
- **I(d)**: apply differencing $d$ times before modelling.
- **MA(q)**: the moving-average-errors piece.

For seasonal data we extend this to **SARIMA**: a non-seasonal $(p, d, q)$
order *and* a seasonal $(P, D, Q)_m$ order. For air passengers the
classical "airline model" is $\text{ARIMA}(0, 1, 1)(0, 1, 1)_{12}$ on
log-transformed data.


In [ ]:
# Fit the airline model on log(air).
airline = ARIMA(log_air, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit()
print(airline.summary().tables[1])


A good ARIMA fit should leave **residuals that look like white noise**,
no remaining autocorrelation, roughly normal distribution. We check both.


In [ ]:
fig = airline.plot_diagnostics(figsize=(11, 6))
plt.tight_layout()
plt.show()


Read the four panels:

- **Standardised residual**: should bounce around zero with no pattern.
- **Histogram + KDE**: should look roughly normal.
- **Normal Q-Q**: points should hug the diagonal.
- **Correlogram**: bars should sit inside the blue band (no leftover autocorrelation).

The airline model passes all four, good sign.


## 4.7 Letting `auto_arima` search for an order

Hand-picking $(p, d, q)$ from ACF/PACF takes practice. `pmdarima.auto_arima`
does a stepwise search using AIC and is a useful first pass, but treat it
as a *starting point*, not a final answer. Always inspect the residuals.

This cell **respects `NB_FAST`** so the test suite doesn't pay the search
cost. Interactive learners get the real search.


In [ ]:
if NB_FAST:
    print("NB_FAST=1: skipping auto_arima search; using the hand-picked airline order.")
    auto_order = (0, 1, 1)
    auto_seasonal = (0, 1, 1, 12)
else:
    from pmdarima import auto_arima

    auto_model = auto_arima(
        log_air,
        start_p=0, start_q=0, max_p=2, max_q=2,
        m=12, seasonal=True,
        d=1, D=1,
        stepwise=True, suppress_warnings=True, error_action="ignore",
        information_criterion="aic", trace=False,
    )
    auto_order = auto_model.order
    auto_seasonal = auto_model.seasonal_order
    print(f"auto_arima chose {auto_order} x {auto_seasonal}")

print(f"order used: {auto_order} x {auto_seasonal}")


## Exercises

**Exercise 4.1**: *Make a series stationary.*
Load the CO2 series from notebook 2:

```python
from statsmodels.datasets import co2
co2_series = co2.load_pandas().data.resample("MS").mean().squeeze().dropna()
```

How many first differences and seasonal differences (lag 12) do you need
before `stationarity_report` agrees the series is stationary? Try
`(d=1, D=0)`, `(d=0, D=1)`, `(d=1, D=1)` and report.

**Exercise 4.2**: *Read the diagnostics.*
For an ARMA model, ACF that decays slowly + PACF that cuts off at lag p
suggests an AR(p). Generate an AR(1) series with $\phi = 0.8$ (n=300), plot
its ACF and PACF, and confirm the rule of thumb by eye.

**Exercise 4.3**: *Fit and compare.*
On `log(air)`, fit **three** SARIMA models, the airline model
$(0,1,1)(0,1,1)_{12}$, $(1,1,0)(0,1,1)_{12}$, and $(1,1,1)(1,1,1)_{12}$.
Print each model's AIC. Which one wins, and is the winner's improvement
"big enough to care" or just noise?

*Hint:* AIC differences of <2 are considered indistinguishable; >10 is decisive.

**Exercise 4.4 (stretch)**: *Forecast with confidence intervals.*
Use `airline.get_forecast(steps=24).summary_frame()` to get point forecasts
*and* 95% prediction intervals for the next two years on the **log** scale,
then exponentiate to get them back on the passenger-count scale. Plot the
forecast band on top of the original `air` series. (We will systematise
this in notebook 5.)


---

*Hyndman, R.J. & Athanasopoulos, G. (2021), Forecasting: Principles and
Practice (3rd ed.), Chapter 9 (ARIMA models). <https://otexts.com/fpp3/>*
